In [35]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
from scipy.signal import welch
from scipy.stats import entropy

In [36]:
from config import dir_config
compiled_dir = dir_config.data.compiled
processed_dir = dir_config.data.processed

### Load subject metadata

In [37]:
data_dir = Path(compiled_dir, "data_100Hz")
subject_metadata = pd.read_csv(Path(compiled_dir, 'participant_info.csv'))

### Clean subject metadata 
check missing, Mean_SaO2 in numbers

In [38]:
# all column names are lower and snake style
subject_metadata.columns = subject_metadata.columns.str.lower().str.replace(' ', '_')

In [39]:
# print column with null
for column, count in subject_metadata.isnull().sum().items():
    if count > 0:
        print(f"Column: {column}, Null Count: {count}")

Column: medical_history, Null Count: 13
Column: sleep_disorders, Null Count: 6


In [40]:
# remove % and convert to integers for mean_saO2
subject_metadata['mean_sao2'] = subject_metadata['mean_sao2'].apply(lambda row: int(row[:-1]))

# convert nan to empty list, str of conditions to list of str
subject_metadata['sleep_disorders'] = (
    subject_metadata['sleep_disorders']
    .fillna('')                              # replace NaN with empty string
    .apply(lambda x: [s.strip() for s in x.split(',')] if x else [])
)

subject_metadata['medical_history'] = (
    subject_metadata['medical_history']
    .fillna('')                              # replace NaN with empty string
    .apply(lambda x: [s.strip() for s in x.split(',')] if x else [])
)

One hot encode Gender, medical history, sleep disorders

In [41]:
# Keep keys normalized (lowercase).
SYN2CAN = {
# sleep apnea family
"sleep apnea": "osa",
"sleep apena": "osa", # just in case
"osa": "osa",
"h/o osa": "osa",
# snoring family
"snore": "snoring",
"snoring": "snoring",
"snorts": "snoring",
"snort": "snoring",
# breathing issues
"difficulty breathing": "dyspnea",
"diffifulty breathing" : "dyspnea",
"difficulty staying asleep": "insomnia",
# teeth grinding
"bruxism": "bruxism",
"grinds teeth": "bruxism",
"grind teeth": "bruxism",
# fatigue / sleepiness
"fatigue": "fatigue",
"chronic fatigue": "fatigue",
"hypersomnia": "hypersomnia",
"eds": "eds", # excessive daytime sleepiness
"rbd": "rbd",
"rls": "rls",
# headaches
"headache": "migraine",
"morning headaches": "migraine",
"migraine": "migraine",
# none
"none": None,
# medical history (normalize to snake-like)
"asthma": "asthma",
"body pain": "body_pain",
"gerd": "gerd",
"anxiety": "anxiety",
"depression": "depression",
"dyspnea": "dyspnea",
"diabetes": "diabetes",
"arrhythmia": "arrhythmia",
"cad": "cad",
"hypertension": "hypertension",
"mci": "mci"
}

def normalize_med_history(history_list, synmap):
    """
    Takes a list of strings in medical_history,
    normalizes via SYN2CAN, removes duplicates and None.
    """
    if not isinstance(history_list, list):
        return []

    normalized = []
    for h in history_list:
        key = h.lower().strip()
        mapped = synmap.get(key, key)   # default: keep same key
        if mapped is not None:
            normalized.append(mapped)

    # dedupe while preserving order
    return list(dict.fromkeys(normalized))

subject_metadata["medical_history_norm"] = subject_metadata["medical_history"].apply(
    lambda x: normalize_med_history(x, SYN2CAN)
)
subject_metadata["sleep_disorders_norm"] = subject_metadata["sleep_disorders"].apply(
    lambda x: normalize_med_history(x, SYN2CAN)
)

In [42]:
gender_encoder = OneHotEncoder(drop='first', sparse_output=False, dtype=int)
gender_ohe = gender_encoder.fit_transform(subject_metadata[['gender']])
df_gender = pd.DataFrame(gender_ohe, columns=gender_encoder.get_feature_names_out(['gender']))
subject_metadata = subject_metadata.join(df_gender)
subject_metadata = subject_metadata.drop('gender', axis=1)

mlb_sleep = MultiLabelBinarizer()
sleep_ohe = mlb_sleep.fit_transform(subject_metadata['sleep_disorders_norm'])
df_sleep_ohe = pd.DataFrame(sleep_ohe, columns=[f"sd_{d.replace(' ', '_')}" for d in mlb_sleep.classes_])
subject_metadata = subject_metadata.join(df_sleep_ohe)
subject_metadata = subject_metadata.drop('sleep_disorders_norm', axis=1)
subject_metadata = subject_metadata.drop('sleep_disorders', axis=1)

mlb_med = MultiLabelBinarizer()
med_ohe = mlb_med.fit_transform(subject_metadata['medical_history_norm'])
df_med_ohe = pd.DataFrame(med_ohe, columns=[f"med_{d.replace(' ', '_')}" for d in mlb_med.classes_])
subject_metadata = subject_metadata.join(df_med_ohe)
subject_metadata = subject_metadata.drop('medical_history_norm', axis=1)
subject_metadata = subject_metadata.drop('medical_history', axis=1)

In [43]:
# all column names are lower and snake style
subject_metadata.columns = subject_metadata.columns.str.lower().str.replace(' ', '_')
print(subject_metadata.shape)
subject_metadata.head()

(100, 33)


,sid,age,bmi,oahi,ahi,mean_sao2,arousal_index,gender_m,sd_bruxism,sd_dyspnea,...,med_asthma,med_body_pain,med_cad,med_depression,med_diabetes,med_dyspnea,med_gerd,med_hypertension,med_migraine,med_osa
0,S002,65.90,27.0,19,19,91,98,1,0,0,...,1,1,0,0,0,0,1,1,0,1
1,S003,29.38,51.0,34,37,95,28,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,S004,55.66,41.0,63,99,89,109,0,0,1,...,0,1,0,1,0,1,1,0,0,0
3,S005,49.12,43.0,19,20,95,28,0,0,0,...,1,1,0,1,1,1,1,0,0,1
4,S006,36.91,22.0,4,5,97,34,0,0,0,...,0,0,0,1,0,0,0,0,0,1


In [44]:
subject_metadata.to_csv(Path(processed_dir, 'participant_info.csv'), index=False)

### Feature Engineering

In [45]:
from src.feature_extraction import run_all

In [ ]:
# Start with max_workers=4 or 6 and scale up
run_all(
    input_dir=str(Path(compiled_dir, "data_100Hz")),
    out_dir=str(Path(processed_dir)),
    epoch_sec= 5, # 5 second epochs
    fs=100, # 100 Hz sampling frequency
)

### Combine metadata

In [ ]:
# load each parquet file, append metadata to each subject df, and concatenate into one big df

agg_data = pd.DataFrame()

parquet_files = list(Path(processed_dir).glob('*.parquet'))

for parquet_file in parquet_files:
    df = pd.read_parquet(parquet_file)
    subject_id = parquet_file.name.strip('.parquet')
    metadata_row = subject_metadata[subject_metadata['sid'] == subject_id]
    if len(metadata_row) == 1:
        metadata_row = metadata_row.drop(columns=['sid'])
        for col in metadata_row.columns:
            df[col] = metadata_row[col].values[0]
    else:
        print(f"Warning: Subject ID {subject_id} not found in metadata or duplicate entries.")
    df.to_parquet(Path(processed_dir, f"{parquet_file.stem}_with_metadata.parquet"))
    agg_data = pd.concat([agg_data, df], ignore_index=True)

# save agg_data to parquet
agg_data.to_parquet(Path(processed_dir, "agg_data_with_metadata.parquet"), index=False)